# 02 — Many Paths

In [`01_`](../01_what_is_a_walk/) we watched a single random walk. One walk is a story — interesting, but you can't generalize from it. *"This walk ended at +18"* tells you nothing about where the next walk will end.

To see *patterns*, we need to look at lots of walks at once. So let's run a thousand of them, side by side, and ask: **what does the set of all possible walks look like?**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

## Generating many walks at once

The walk in `01_` was a 1D array of 1000 numbers. To do *many* walks, we just stack them as rows of a 2D array. Same logic, one extra dimension.

In [ ]:
N_WALKS = 500
N_STEPS = 1000
SEED = 42

rng = np.random.default_rng(SEED)

# Shape: (N_WALKS, N_STEPS) — each row is one walk's sequence of ±1 steps.
steps = rng.choice([-1, 1], size=(N_WALKS, N_STEPS))

# cumsum along axis=1 gives each walk's running total over time.
# Prepend a column of zeros so every walk starts at 0.
positions = np.concatenate(
    [np.zeros((N_WALKS, 1), dtype=int), np.cumsum(steps, axis=1)],
    axis=1,
)

print(f"positions shape: {positions.shape}  (rows = walks, cols = time)")
print()
print(f"final positions — min: {positions[:, -1].min()}, max: {positions[:, -1].max()}")
print(f"final positions — mean: {positions[:, -1].mean():.2f}  (expected near 0)")
print(f"final positions — std:  {positions[:, -1].std():.2f}  (we'll come back to this)")
print(f"sqrt(N_STEPS):          {np.sqrt(N_STEPS):.2f}")

### What just happened

- `rng.choice([-1, 1], size=(500, 1000))` — 500,000 coin flips arranged as a 500-by-1000 grid. NumPy generates them all at once, vectorized, in a fraction of a second.
- `np.cumsum(steps, axis=1)` — running total along each row. Now each row is one walk's position over time.
- We prepend a column of zeros so every walk starts at the origin.

The most interesting line of the output: the standard deviation of the final positions is suspiciously close to $\sqrt{1000}$. Hold onto that — it's the punchline of the next notebook.

## The fan

Let's plot all 500 walks on the same axes, with very low transparency so the *density* of paths becomes visible. Where many walks pass through the same region, that region gets dark. Where few walks reach, it stays light.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(positions.T, linewidth=0.5, alpha=0.05, color="steelblue")
ax.axhline(0, color="gray", linewidth=0.6, linestyle="--", alpha=0.7)
ax.set_xlabel("step number (t)")
ax.set_ylabel("position")
ax.set_title(f"{N_WALKS} random walks, all starting at 0")
ax.grid(alpha=0.3)
plt.show()

## What to notice

**1. They all start at a single point and fan outward.** The "fan" gets wider over time — but not linearly. It widens fast at first, then slower.

**2. The dense core lives near zero.** Most walks spend most of their time within ±30 or so of the origin, even after 1000 steps. Only a few outliers reach the edges of the picture.

**3. The shape of the fan is roughly symmetric.** Equal density above and below zero. That makes sense — the coin is fair, so there's no reason walks should prefer one side.

**4. No walk gets anywhere close to ±1000.** Even the most extreme outliers barely reach ±100. The vast majority stay within ±50.

The fan shape is the visual signature of a **distribution evolving in time**. At any fixed step `t`, the cross-section of the fan is the distribution of where walks are at that moment.

## Watch the fan widen

Static is fine, but the *widening* is the interesting part. Here we animate 100 walks growing simultaneously. (We use fewer than 500 for the animation so it stays smooth.)

In [ ]:
N_WALKS_ANIM = 100
positions_anim = positions[:N_WALKS_ANIM]

STEPS_PER_FRAME = 10
n_frames = N_STEPS // STEPS_PER_FRAME

fig, ax = plt.subplots(figsize=(10, 5))
ax.set_xlim(0, N_STEPS)
ax.set_ylim(positions.min() - 5, positions.max() + 5)
ax.axhline(0, color="gray", linewidth=0.6, linestyle="--", alpha=0.7)
ax.set_xlabel("step number (t)")
ax.set_ylabel("position")
ax.set_title(f"watching {N_WALKS_ANIM} walks unfold together")
ax.grid(alpha=0.3)

lines = [
    ax.plot([], [], linewidth=0.7, alpha=0.25, color="steelblue")[0]
    for _ in range(N_WALKS_ANIM)
]

def update(frame):
    end = (frame + 1) * STEPS_PER_FRAME + 1
    x_data = np.arange(end)
    for i, line in enumerate(lines):
        line.set_data(x_data, positions_anim[i, :end])
    return lines

ani = FuncAnimation(fig, update, frames=n_frames, interval=40, blit=True)
plt.close(fig)
HTML(ani.to_jshtml())

## Where do they end up?

We've been looking at the fan as paths through time. Now let's freeze time at the end (`t = 1000`) and ask: where did the 500 walks end up? A histogram of the final positions tells us.

In [ ]:
final_positions = positions[:, -1]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(final_positions, bins=40, color="steelblue", alpha=0.8, edgecolor="black")
ax.axvline(0, color="red", linestyle="--", alpha=0.7, label="start (0)")
ax.axvline(final_positions.mean(), color="black", linestyle="-",
           alpha=0.7, label=f"mean = {final_positions.mean():.1f}")
ax.set_xlabel(f"final position after {N_STEPS} steps")
ax.set_ylabel("number of walks")
ax.set_title(f"distribution of final positions across {N_WALKS} walks")
ax.legend()
plt.show()

That shape is a **bell curve**.

This isn't an accident or a numerical coincidence — it's one of the deepest results in probability, the **Central Limit Theorem**. The sum of many small independent random pieces (here: 1000 coin flips) tends toward a Gaussian distribution, regardless of what the individual pieces look like. We'll meet the CLT properly later. For now, just notice that it shows up *automatically*, just from running the simulation.

## How the distribution evolves over time

Here's the picture that ties it all together. Three snapshots of the distribution — at t = 100, t = 500, and t = 1000 — drawn on the same x-axis so you can directly compare their widths.

In [ ]:
times_to_show = [100, 500, 1000]

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True)
for ax, t in zip(axes, times_to_show):
    snapshot = positions[:, t]
    ax.hist(snapshot, bins=30, color="steelblue", alpha=0.8, edgecolor="black")
    ax.axvline(0, color="red", linestyle="--", alpha=0.5)
    ax.set_title(f"t = {t}\nstd = {snapshot.std():.1f},  $\\sqrt{{t}}$ = {np.sqrt(t):.1f}")
    ax.set_xlabel("position")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("number of walks")
plt.tight_layout()
plt.show()

print("standard deviation of position vs sqrt(t):")
for t in times_to_show:
    print(f"  t = {t:>4}:  std = {positions[:, t].std():>6.2f}   sqrt(t) = {np.sqrt(t):.2f}")

## The takeaway

Look at the std-vs-sqrt(t) numbers. They match almost perfectly.

- At t = 100, std ≈ √100 = 10
- At t = 500, std ≈ √500 ≈ 22
- At t = 1000, std ≈ √1000 ≈ 32

The distribution at any time `t` is approximately a bell curve centered at 0, with width about $\sqrt{t}$. That's the **$\sqrt{t}$ law**. It's why the fan widens — and why it widens *slowly*. To double the width, you need to *quadruple* the time.

This was visible in the single walk in `01_` (it never strayed far from zero), and now it's quantitatively obvious in the distribution. In [`03_sqrt_t_law/`](../03_sqrt_t_law/) we'll derive *why* this works mathematically, instead of just observing it.

## Try this

- **Crank up `N_WALKS`** to 5000 or 10000. The histogram gets smoother and the bell shape becomes unmistakable.
- **Drop `N_STEPS`** to 50 or 100. With few steps, the distribution is "lumpy" — only a finite number of distinct final positions are reachable. The bell shape only emerges as the number of steps grows.
- **Change the times in `times_to_show`** to e.g. `[10, 100, 1000]`. Watch how dramatically the spread grows on a log time scale (since √t grows much slower than t itself).
- **Bias the coin** by passing `p=[0.45, 0.55]` to `rng.choice(...)`. Now there's a slight rightward push. What happens to the mean and the spread? Does the distribution still look like a bell?